# Fine tune with the PCAWG dataset

In [1]:
import pandas as pd
import numpy as np
from monte import Monte
from copy import deepcopy
from scipy.stats import pearsonr
from sklearn.model_selection import train_test_split, KFold
from matplotlib import pyplot as plt
from monte import fine_tune_with_cv

In [2]:
cancer_types = ["ov_au", "paca_au", "prad_ca"]

In [3]:
# for cancer in cancer_types:
#     model = Monte.load("../../data/monte_outputs/trained_models/monte_pancancer_model.pkl")
    
#     beta = pd.read_parquet(f"../../data/cancer-methyl/external/pcawg/pcawg_{cancer}_beta.parquet").T
#     meta = pd.read_parquet(f"../../data/cancer-methyl/external/pcawg/pcawg_{cancer}_meta.parquet")

#     beta.dropna(axis=1, inplace=True)
#     meta.index = meta["donor_id"]
#     meta = meta.loc[beta.index]

#     print(f"{cancer}, {beta.shape[0]}")
#     # before fine-tuning
#     b_purity = model.predict_purity(beta)
#     meta["before_finetuning_purity"] = b_purity

#     # after fine-tuning
#     ft_model = fine_tune_with_cv(model, beta, meta["purity"])
#     a_purity = ft_model.predict_purity(beta)
#     meta["after_finetuning_purity"] = a_purity

#     meta.to_csv(f"../../data/external_dataset/pcawg_{cancer}_purity_predictions.csv", index=False)

In [ ]:
n_splits = 5
random_state = 42

for cancer in cancer_types:
    base_model = Monte.load("../../data/monte_outputs/trained_models/monte_pancancer_model.pkl")

    # the parquet file and metadata csv file could be downloaded from Zenodo
    beta = pd.read_parquet(f"../../data/cancer-methyl/external/pcawg/pcawg_{cancer}_beta.parquet").T
    meta = pd.read_parquet(f"../../data/cancer-methyl/external/pcawg/pcawg_{cancer}_meta.parquet")

    beta.dropna(axis=1, inplace=True)

    meta.index = meta["donor_id"]
    meta = meta.loc[beta.index]

    print(f"{cancer}, {beta.shape[0]} samples")

    meta["before_finetuning_purity"] = base_model.predict_purity(beta)
    oof_pred = pd.Series(index=beta.index, dtype=float)

    kf = KFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state
    )

    for fold, (train_idx, test_idx) in enumerate(kf.split(beta), start=1):
        print(f"  Fold {fold}/{n_splits}")

        train_samples = beta.index[train_idx]
        test_samples = beta.index[test_idx]

        beta_train = beta.loc[train_samples]
        beta_test = beta.loc[test_samples]

        y_train = meta.loc[train_samples, "purity"]

        # Load a fresh base model for each fold
        # so folds do not contaminate each other
        fold_model = Monte.load(
            "../../data/monte_outputs/trained_models/monte_pancancer_model.pkl"
        )

        ft_model = fine_tune_with_cv(
            fold_model,
            beta_train,
            y_train
        )

        oof_pred.loc[test_samples] = ft_model.predict_purity(beta_test)

    meta["after_finetuning_purity"] = oof_pred.loc[meta.index].values

    meta.to_csv(f"../../data/external_dataset/pcawg_{cancer}_purity_predictions.csv", index=False)

ov_au, 70 samples
  Fold 1/5
  Fold 2/5
  Fold 3/5
  Fold 4/5
  Fold 5/5
paca_au, 93 samples
  Fold 1/5
  Fold 2/5
  Fold 3/5
  Fold 4/5
  Fold 5/5
prad_ca, 98 samples
  Fold 1/5
  Fold 2/5
  Fold 3/5
  Fold 4/5
  Fold 5/5


## Sample subset performance

In [4]:
beta = pd.read_parquet(f"../../data/cancer-methyl/external/pcawg/pcawg_{cancer}_beta.parquet").T
meta = pd.read_parquet(f"../../data/cancer-methyl/external/pcawg/pcawg_{cancer}_meta.parquet")

In [6]:
results = []
for cancer in ["ov_au", "paca_au", "prad_ca"]:
    for train_ratio in [x / 10 for x in range(0, 11)]:
        for _seed in range(10):
            beta = pd.read_parquet(f"../../data/cancer-methyl/external/pcawg/pcawg_{cancer}_beta.parquet").T
            beta.dropna(axis=1, inplace=True)
            meta = pd.read_parquet(f"../../data/cancer-methyl/external/pcawg/pcawg_{cancer}_meta.parquet")
            meta.index = meta["donor_id"]
            meta = meta.loc[beta.index, :]
            beta_train, beta_test, meta_train, meta_test = train_test_split(beta, meta, test_size=0.3, random_state=42)

            model = Monte.load("../../data/monte_outputs/trained_models/monte_pancancer_model.pkl")
            if train_ratio == 0.0:
                pass
            elif train_ratio == 1.0:
                model = model.fine_tune(beta_train, meta_train["purity"])
            else:
                beta_train, _, meta_train, _ = train_test_split(beta_train, meta_train, test_size=1-train_ratio, random_state=_seed)
                model = model.fine_tune(beta_train, meta_train["purity"])
            
            predicted_purity = model.predict_purity(beta_test)
            corr, pvalues = pearsonr(predicted_purity, meta_test["purity"])
            mse = ((predicted_purity - meta_test["purity"]) ** 2).mean()
            if train_ratio == 0.0:
                train_size = 0
            else:
                train_size = beta_train.shape[0]
            results.append([cancer, train_ratio, train_size, beta_test.shape[0], corr, pvalues, mse])

In [9]:
results_df = pd.DataFrame(results, columns=["cancer", "train_ratio", "train_size", "test_size", "pearson_corr", "p_value", "mse"])

In [8]:
results_df.to_csv("../../data/external_dataset/pcawg_fine_tuning_sample_size_results.csv", index=False)